# Kestrel V3, step 4: continued pretraining on quality tokens

`Run all` does whichever stage is next, and every stage resumes after a disconnect.

| stage | runtime | what | cost |
|---|---|---|---|
| A | **CPU** | tests, then ~1.37B unseen tokens (`scripts/prepare_quality.py`), saved to Drive | ~0.5 unit |
| B | **A100** | 7,600 steps = 1.49B tokens from `runs/mtp_control` | ~165 units, ~25 hours |
| C | **A100** | `eval_generalization` on the same suite as every earlier number | ~1 unit |

Stage A stops the notebook when it's done. Switch the runtime to A100 and `Run all` again.


In [ ]:
# 0. Which runtime?
import os, json, shutil, glob, subprocess, time
import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print('torch', torch.__version__, '|', GPU or 'CPU runtime')
if GPU and 'A100' not in GPU and 'H100' not in GPU:
    raise SystemExit(f'Got {GPU}. Use a CPU runtime for stage A, or an A100 for stages B and C.')


In [ ]:
# 1. Drive. The first mount right after the permission prompt often fails with 'mount failed'.
from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError as e:
    print('first mount failed, retrying:', e)
    drive.mount('/content/drive', force_remount=True)
KD = '/content/drive/MyDrive/Kestrel'
assert os.path.isdir(KD), f'{KD} not found - mounted the wrong Google account?'
QD = f'{KD}/datasets/quality_v3'
RUN = f'{KD}/runs/pretrain_v3'
BASE = f'{KD}/runs/mtp_control/ckpt.pt'
assert os.path.exists(BASE), BASE
print('corpus built:', os.path.exists(f'{QD}/manifest.json'), '| run started:', os.path.exists(f'{RUN}/ckpt.pt'))


In [ ]:
# 2. Code (public repo) and its checks. A failed test stops here, before anything is spent.
REPO = '/content/Kestrel'
if not os.path.isdir(REPO):
    !git clone -q https://github.com/YoMosa2009/Kestrel.git {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1
!pip -q install tokenizers datasets
r = subprocess.run(['python', '-m', 'scripts.test_step4'], capture_output=True, text=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': ''})
print(r.stdout[-600:], r.stderr[-1500:])
assert 'STEP4_TEST_OK' in r.stdout, 'step 4 tests failed - see above'


In [ ]:
# A. The corpus (CPU runtime). A 0.1% build first proves every source can be read end to end,
#    then the real one, which copies each finished source to Drive.
if not os.path.exists(f'{QD}/manifest.json'):
    if GPU:
        raise SystemExit('The corpus is not on Drive yet. Stage A is CPU work: switch to a CPU runtime.')
    if not os.path.exists(f'{QD}/progress.json'):
        !python -u -m scripts.prepare_quality --out /content/q_smoke --scale 0.001 --val-tokens 20000
        assert os.path.exists('/content/q_smoke/manifest.json'), 'smoke build failed - see above'
        print(open('/content/q_smoke/manifest.json').read())
    !python -u -m scripts.prepare_quality --out /content/q --sync-dir {QD}
    assert os.path.exists(f'{QD}/manifest.json'), 'build did not finish - Run all again to resume.'
    drive.flush_and_unmount()
if not GPU:
    raise SystemExit('STAGE_A_DONE. Switch the runtime to A100 and Run all for stages B and C.')


In [ ]:
# B. Training. data_5b supplies the skills the fresh data doesn't cover. --schedule-start keeps one
#    LR curve across disconnects; the new domains' val shards show learning on unseen text.
TOTAL_STEPS = 19940 + 7600
MIX = 'qknow=33,qweb=23,qmath=12,qcode=7,code=8,inst=6,cli=3,repo=3,tool=3,docs=1,sec=1'
def finished():
    p = f'{RUN}/status.json'
    return os.path.exists(p) and json.load(open(p)).get('state') == 'done'
def local(src, dst):
    os.makedirs(dst, exist_ok=True)
    for f in sorted(glob.glob(f'{src}/*.bin')):
        out = f'{dst}/{os.path.basename(f)}'
        if not os.path.exists(out) or os.path.getsize(out) != os.path.getsize(f):
            shutil.copy(f, out)
t = time.time()
local(f'{KD}/datasets/corpus-5b', '/content/data_5b')
local(QD, '/content/data_q')
os.makedirs('/content/data_v3', exist_ok=True)
for f in glob.glob('/content/data_5b/*.bin') + glob.glob('/content/data_q/*.bin'):
    link = f'/content/data_v3/{os.path.basename(f)}'
    if not os.path.exists(link):
        os.symlink(f, link)
print(f'data: {sum(os.path.getsize(p) for p in glob.glob("/content/data_v3/*.bin"))/1e9:.2f} GB, {time.time()-t:.0f}s')
os.makedirs(RUN, exist_ok=True)
if not finished():
    get_ipython().system(
        f'python -u -m kestrel.train --preset nano --data-dir /content/data_v3 --curriculum --mix {MIX} '
        f'--bf16 --grad-checkpoint --fused-adam --prefetch 3 --seq 4096 --batch 16 --accum 3 '
        f'--steps {TOTAL_STEPS} --schedule-start 19940 --warmup 30 --decay-frac 0.3 '
        f'--muon-lr 0.005 --adamw-lr 0.00075 --rope-scale 4 --init-from {BASE} --out {RUN} '
        f'--log-every 20 --eval-every 500 --eval-iters 6 --ckpt-every 1000 --ckpt-secs 3600')
assert finished(), 'training did not finish. Run all again to resume.'
print('STAGE_B_DONE')


In [ ]:
# C. val_mean on data_5b alone, the suite behind every earlier number. Baseline: control, 1.6345.
GEN = f'{KD}/evals/step4_generalization.json'
if not os.path.exists(GEN):
    !python -m scripts.eval_generalization --ckpt {RUN}/ckpt.pt --data-dir /content/data_5b --seq 4096 --label step4 --out {GEN}
print(open(GEN).read()[-1800:])
print('STAGE_C_DONE')
